# Integrated Gradients

In [4]:
# from transformers import AutoModelForSequenceClassification, AutoTokenizer
# from transformers_interpret import SequenceClassificationExplainer
# import pandas as pd
# from sklearn.preprocessing import LabelEncoder

# model_name = "C:/Users/anouk.xs4all/Projects/XS4ALL/saved_models/v2_3_entrtrian_loss_skf_fold4"
# model = AutoModelForSequenceClassification.from_pretrained(model_name)
# tokenizer = AutoTokenizer.from_pretrained("roberta-base")

# df = pd.read_json(r"C:\Users\anouk.xs4all\Projects\XS4ALL\training\train_w13.jsonl", lines=True)

# label_encoder = LabelEncoder()
# df['label'] = label_encoder.fit_transform(df['period_bucket'])

Loading weights: 100%|█████████████████████████████████████████████████████████████| 201/201 [00:00<00:00, 1639.52it/s]


<div class="alert alert-block alert-danger">
<b>WARNING: Running the following cells will lead to severe data loss</b>. The datasets used for this analysis are protected property of the KB National Library in the Netherlands. Therefore, paths to the user's annotations of the application are provided if one would like to test the analysis by themselves. 

However, <b> if you are interested in a deep dive of the XAI analysis of this project, do NOT run the following cells. </b>
</div>

![](img/confusion_matrix.v2.3_optimal_weights.png)

In [3]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from transformers_interpret import SequenceClassificationExplainer
import pandas as pd
from sklearn.preprocessing import LabelEncoder
import torch

/Users/anoukflinkert/TIME2WARC/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_v2_3_name = "/Users/anoukflinkert/TIME2WARC/models/v2_3_optimal_weights_fold1"
model_v2_3 = AutoModelForSequenceClassification.from_pretrained(model_v2_3_name)
model_v2_3.eval()

tokenizer_v2_3 = AutoTokenizer.from_pretrained("roberta-base")

# # Make sure to use the correct dataset for each model
df_12 = pd.read_json(r"<path_to_your_jsonl_file>", lines=True)

label_encoder = LabelEncoder()
df_12['label'] = label_encoder.fit_transform(df_12['period_bucket'])

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 7091.23it/s]


In [ ]:
# After a few test runs some generators still showed,  but that is probably due to varying notations of e.g. 'FrontPage.Editor'

import sys
sys.path.append("<path_to_your_script_training_directory>")
from labeling_function_v8 import SignalExtractor

# Hard-coding the masking, already defined in the model algorithm but just to be sure.
def mask_payload(payload):
    if not isinstance(payload, str):
        return ""

    for signal_name, pattern in SignalExtractor.regex_patterns.items():
        payload = pattern.sub("", payload)
    return payload

In [6]:
# Testing the masking
subset = df_12[df_12['label'] == 0]
subset_reset = subset.reset_index(drop=True)
samples = []
for idx, row in subset[10:15].iterrows():
    subset_reset['payload'] = mask_payload(row['payload'])
    samples.append({
        # 'url': subset_reset.iloc[idx].get('url', 'unknown'),
        # 'payload_preview': subset_reset.iloc[idx]['payload'][:500],
        'idx': idx
    })

print(samples)

[{'idx': 77}, {'idx': 78}, {'idx': 83}, {'idx': 90}, {'idx': 101}]


### 3.1 Targeting the misclassifications cases

In [14]:
from tqdm import tqdm

def missclassifications(model, tokenizer, df, true_label_idx, miss_label_idx, max_search=200):
    id2label = {0: "19971999", 1: "20002002", 2: "20032006", 3: "20072010"}
    
    true_str = id2label.get(true_label_idx)
    miss_str = id2label.get(miss_label_idx)

    # print("Searching for misclassifications...")

    subset = df[df['label'] == true_label_idx]

    count = 0

    subset_reset = subset.reset_index(drop=True)
    samples = []
    for idx, row in tqdm(subset_reset.iterrows(), total=len(subset_reset), desc="Calculating wrong predictions"):
        row['payload'] = mask_payload(row['payload'])
        
        if idx > max_search:
            print("Reached limit without finding more")
            break
            
        inputs = tokenizer(row['payload'], return_tensors="pt", truncation=True, max_length=512).to(model.device)

        # print(f"Test: {row['payload'][:200]}")
        
        with torch.no_grad():
            outputs = model(**inputs)
            
        predicted_idx = outputs.logits.argmax().item()

        # Check if this specific instance hits the error group
        if predicted_idx == miss_label_idx:
            samples.append({
                'url': subset_reset.iloc[idx].get('url', 'unknown'),
                # 'idx': idx,
                'payload': row['payload'],
                'true_label' : true_str,
                'pred_label' : id2label.get(predicted_idx)
            })

            if len(samples) >=5:
                break
    
    return samples

In [148]:
# We will load them again in the case studies below
# conf_97_03 = missclassifications(model_v2_3, tokenizer_v2_3, df_13, 0, 2)
# conf_97_07 = missclassifications(model_v2_3, tokenizer_v2_3, df_13, 0, 3)
# conf_03_97 = missclassifications(model_v2_3, tokenizer_v2_3, df_13, 2, 0)

In [149]:
def test(subset):
    item = subset[0]
    warc_payload = item['payload'] 
    url = item['url']
    return (warc_payload, 
            url, 
            print("--- Top Positive Tokens ---"), 
            print("=== Top Positive Tokens ==="))

print(test(conf_97_03))

--- Top Positive Tokens ---
=== Top Positive Tokens ===
('\n<!-- last modified on 07-01-2000 -->\n<html>\n<head>\n<meta name="Generator" content="UDO6 PL14 for TOS">\n<meta name="Keywords" content="Atari, ST, TT, Falcon, Hades, Milan, MagicMac">\n<meta name="Description" content="Stichting ST, Atari, Typefaces for the Atari">\n<meta name="Email" content="dziewon@xs4all.nl">\n<link rev=made href="mailto:dziewon@xs4all.nl" title="E-Mail">\n<link rel=home href="index.htm" title="Homepage">\n<title>Stichting ST - de Thuishaven van Atari in Nederland</title>\n</head>\n<frameset cols="30%,*" border=0 frameborder=0 framespacing=0>\n\t<frame src="00toc.htm" name="UDOtoc" marginwidth=0 marginheight=0>\n\t<frame src="00con.htm" name="UDOcon" marginwidth=0 marginheight=0>\n</frameset>\n</html>\n\n\n\n<!-- last modified on 07-01-2000 -->\n<html>\n<head>\n<title>Inhoud</title>\n<meta name="Generator" content="UDO6 PL14 for TOS">\n<meta name="Email" content="dziewon@xs4all.nl">\n<link re', 'www.atar

### Denoising token fragmentation

In [8]:
import re
import string
import numpy as np

# ==========================================
# 1. NOISE FILTERING
# ==========================================
noise_tokens = {
    '<s>', '</s>', '<pad>', '<unk>', '<mask>',
    *list(string.printable)
}

def is_noise_token(token):
    # Remove special RoBERTa prefix
    clean = token.replace('Ġ', '').replace('Ċ', '').strip()
    
    if not clean:
        return True
    if clean in noise_tokens:
        return True
    if re.match(r'^\d+$', clean):  # filter isolated numbers
        return True
    if len(clean) <= 1:
        return True

    return False


# ==========================================
# 2. INDEX-AWARE BPE MERGING
# ==========================================
def merge_bpe_attributions_with_indices(tokens, attributions):
    """
    Merges sub-tokens and averages their attributions, while explicitly tracking 
    the original list indices of every sub-token for precise downstream masking.
    """
    merged_tokens, merged_attrs, merged_indices = [], [], []
    current_token, current_attrs, current_idx_group = "", [], []
    
    for i, (token, attr) in enumerate(zip(tokens, attributions)):
        clean_token = token.replace('Ġ', ' ').replace('Ċ', '')
        
        if token.startswith('Ġ') or token.startswith('<'):
            if current_token:
                merged_tokens.append(current_token)
                merged_attrs.append(np.mean(current_attrs))
                merged_indices.append(current_idx_group)
            
            # Start a new token buffer
            current_token = clean_token.lstrip() 
            current_attrs = [attr]
            current_idx_group = [i]
        else:
            # Append to existing token buffer
            current_token += clean_token
            current_attrs.append(attr)
            current_idx_group.append(i)
            
    # Flush the final buffer
    if current_token:
        merged_tokens.append(current_token)
        merged_attrs.append(np.mean(current_attrs))
        merged_indices.append(current_idx_group)
        
    return merged_tokens, merged_attrs, merged_indices

### 3.2 Visualizations and manual inspections

In [9]:
# ==========================================
# 3. MAIN VISUALIZATION PIPELINE
# ==========================================
def visualization(model, tokenizer, subset, item_index=0):
    cls_explainer = SequenceClassificationExplainer(model, tokenizer)
    model.config.id2label = {0: "1997-1999", 1: "2000-2002", 2: "2003-2006", 3: "2007-2010"}

    item = subset[item_index]
    warc_payload = item['payload']

    # ===== ORIGINAL PAYLOAD EXPLANATION =====
    attributions = cls_explainer(warc_payload)

    tokens = [a[0] for a in attributions]
    scores = [a[1] for a in attributions]

    # Merge tokens and retrieve their absolute indices
    merged_tokens, merged_scores, merged_indices = merge_bpe_attributions_with_indices(tokens, scores)

    # Build dataframe with the Index tracker column
    attr_df = pd.DataFrame({
        'Token': merged_tokens, 
        'Score': merged_scores,
        'Indices': merged_indices
    })
    
    # Filter noise and reset index
    attr_df = attr_df[~attr_df['Token'].apply(is_noise_token)].reset_index(drop=True)

    # Extract the top 10 positive tokens and their target indices
    top_10_df = attr_df.sort_values(by='Score', ascending=False).head(10)
    top_tokens = top_10_df['Token'].tolist()
    
    # Flatten the list of lists into a single list of target indices
    indices_to_mask = []
    for idx_group in top_10_df['Indices']:
        indices_to_mask.extend(idx_group)

    print(f"Top 10 attributed tokens: {top_tokens}")

    # ===== MASKED PAYLOAD GENERATION (INDEX PROTOCOL) =====
    # 1. Tokenize the original payload to get the exact tensor array
    inputs = tokenizer(warc_payload, truncation=True, max_length=512)
    input_ids = inputs['input_ids']

    # 2. Iterate through our target indices and overwrite them with the model's standard <mask> ID
    for idx in indices_to_mask:
        # Failsafe check to ensure index is within bounds of truncated sequence
        if idx < len(input_ids): 
            input_ids[idx] = tokenizer.mask_token_id

    # 3. Decode the exact tensor array back into a text string for the explainer to read.
    # We strip the <s> and </s> tokens ([1:-1]) so the explainer doesn't double-wrap the sequence.
    masked_payload = tokenizer.decode(input_ids[1:-1])


    # ===== RUN EXPLAINER ON MASKED PAYLOAD =====
    attributions_masked = cls_explainer(masked_payload)
    
    tokens_masked = [a[0] for a in attributions_masked]
    scores_masked = [a[1] for a in attributions_masked]

    # We don't need indices for the masked visualization DataFrame
    merged_tokens_masked, merged_scores_masked, _ = merge_bpe_attributions_with_indices(tokens_masked, scores_masked)
    
    attr_df_masked = pd.DataFrame({'Token': merged_tokens_masked, 'Score': merged_scores_masked})
    attr_df_masked = attr_df_masked[~attr_df_masked['Token'].apply(is_noise_token)].reset_index(drop=True)

    # ===== PRINT RESULTS =====
    print(f"\nURL: {item['url']}")
    print(f"True label: {item['true_label']}")

    print("\n=== ORIGINAL PAYLOAD ===")
    print("Top Positive Tokens:")
    print(attr_df[['Token', 'Score']].sort_values(by='Score', ascending=False).head(10))
    print("\nTop Negative Tokens:")
    print(attr_df[['Token', 'Score']].sort_values(by='Score', ascending=True).head(10))

    print("\n=== MASKED PAYLOAD ===")
    print("Top Positive Tokens:")
    print(attr_df_masked.sort_values(by='Score', ascending=False).head(10))
    print("\nTop Negative Tokens:")
    print(attr_df_masked.sort_values(by='Score', ascending=True).head(10))

    # ===== HTML VISUALIZATIONS =====
    # Restore state on original payload
    cls_explainer(warc_payload)
    viz_original = cls_explainer.visualize(true_class=item['true_label'])

    # Update state on masked payload
    cls_explainer(masked_payload)
    viz_masked = cls_explainer.visualize(true_class=item['true_label'])

    return viz_original, viz_masked, top_tokens, attr_df, attr_df_masked

#### 3.2.1 1997–1999 $\leftrightarrow$ 2003–2006
Why does a 90s website look like the late 2000s to RoBERTa?

In [ ]:
# Load the sample subset (n=3)
conf_97_03 = missclassifications(model_v2_3, tokenizer_v2_3, df_12, 0, 2)

Calculating wrong predictions:   6%|██▉                                               | 27/455 [00:22<05:53,  1.21it/s]


In [45]:
# Visualize the attribution scores of the first sample (change item_index) 
visualization(model_v2_3, tokenizer_v2_3, conf_97_03, item_index=0)

Top 10 attributed tokens: ['<!--lastmodifiedon07-01-2000-->', '<metaname="Generator"content="UDO6PL14forTOS">', '<head>', '<metaname="Generator"content="UDO6PL14forTOS">', '</title>', '<html>', '<html>', '<metaname="Description"content="StichtingST,Atari,TypefacesfortheAtari">', '</head>', '<metaname="Keywords"content="Atari,ST,TT,Falcon,Hades,Milan,MagicMac">']

URL: www.atarist.home.xs4all.nl
True label: 19971999

=== ORIGINAL PAYLOAD ===
Top Positive Tokens:
                                                Token     Score
0                     <!--lastmodifiedon07-01-2000-->  0.089493
3      <metaname="Generator"content="UDO6PL14forTOS">  0.059825
19                                             <head>  0.054798
22     <metaname="Generator"content="UDO6PL14forTOS">  0.028110
10                                           </title>  0.015160
18                                             <html>  0.014341
1                                              <html>  0.010682
5   <metaname="Descrip

(<IPython.core.display.HTML object>,
 <IPython.core.display.HTML object>,
 ['<!--lastmodifiedon07-01-2000-->',
  '<metaname="Generator"content="UDO6PL14forTOS">',
  '<head>',
  '<metaname="Generator"content="UDO6PL14forTOS">',
  '</title>',
  '<html>',
  '<html>',
  '<metaname="Description"content="StichtingST,Atari,TypefacesfortheAtari">',
  '</head>',
  '<metaname="Keywords"content="Atari,ST,TT,Falcon,Hades,Milan,MagicMac">'],
                                                 Token     Score  \
 0                     <!--lastmodifiedon07-01-2000-->  0.089493   
 1                                              <html>  0.010682   
 2                                              <head>  0.004659   
 3      <metaname="Generator"content="UDO6PL14forTOS">  0.059825   
 4   <metaname="Keywords"content="Atari,ST,TT,Falco...  0.007674   
 5   <metaname="Description"content="StichtingST,At...  0.009073   
 6       <metaname="Email"content="dziewon@xs4all.nl"> -0.000974   
 7   <linkrev=madehref=

In [18]:
visualization(model_v2_3, tokenizer_v2_3, conf_97_03, item_index=4)

Top 10 attributed tokens: ['</a>', '<html>', '<li><ahref="keynotes.html">AppleKeynotes', '<!--scp~/my/WWW/index.htmlator0893@xs2:WWW-->', '</html>', '<!--scp~/my/WWW2/index.htmlator0893@xs2:WWW-->', '<!--scp~/my/WWW2/keynotes.htmlator0893@xs2:WWW-->', '<head>', '<html>', '<head>']

URL: www.ator0893.home.xs4all.nl
True label: 19971999

=== ORIGINAL PAYLOAD ===
Top Positive Tokens:
                                                Token     Score
36                                               </a>  0.079134
17                                             <html>  0.045135
10           <li><ahref="keynotes.html">AppleKeynotes  0.038662
2       <!--scp~/my/WWW/index.htmlator0893@xs2:WWW-->  0.030966
16                                            </html>  0.016031
19     <!--scp~/my/WWW2/index.htmlator0893@xs2:WWW-->  0.015686
20  <!--scp~/my/WWW2/keynotes.htmlator0893@xs2:WWW-->  0.015077
1                                              <head>  0.013647
0                                       

(<IPython.core.display.HTML object>,
 <IPython.core.display.HTML object>,
 ['</a>',
  '<html>',
  '<li><ahref="keynotes.html">AppleKeynotes',
  '<!--scp~/my/WWW/index.htmlator0893@xs2:WWW-->',
  '</html>',
  '<!--scp~/my/WWW2/index.htmlator0893@xs2:WWW-->',
  '<!--scp~/my/WWW2/keynotes.htmlator0893@xs2:WWW-->',
  '<head>',
  '<html>',
  '<head>'],
                                                 Token     Score  \
 0                                              <html>  0.012771   
 1                                              <head>  0.013647   
 2       <!--scp~/my/WWW/index.htmlator0893@xs2:WWW-->  0.030966   
 3                   <title>AppleKeynotes;PrincessTutu  0.003930   
 4                                            </title>  0.008599   
 5                                             </head>  0.010225   
 6                                              <body> -0.000617   
 7                      <h1>AppleKeynotes;PrincessTutu  0.010461   
 8                                    

In [11]:
visualization(model_v2_3, tokenizer_v2_3, conf_97_03, item_index=2)

Top 10 attributed tokens: ['</title>', '<bodybgcolor="#FFFFFF"background="foto/tile9.gif"link="#FFFFFF"vlink="#FFFFFF"alink="#FFFFFF"leftmargin="5"topmargin="5"onLoad="">', '</head>', '<metahttp-equiv="Content-Type"content="text/html;charset=utf-8">', '<HTML>', '<head>', '<divalign="center">', '<scripttype="text/javascript"src="stm31.js"></script>', '<divalign="center"><ahref="http://www.nedstat.nl/cgi-bin/viewstat?name=telkas"target="_new"><imgsrc="http://www.nedstat.nl/cgi-bin/nedstat.gif?name=telkas"border="0"width="16"height="16"nosave></a>', '<tdheight="179">']

URL: www.kgdewijn.home.xs4all.nl
True label: 19971999

=== ORIGINAL PAYLOAD ===
Top Positive Tokens:
                                                Token     Score
3                                            </title>  0.068092
9   <bodybgcolor="#FFFFFF"background="foto/tile9.g...  0.033583
8                                             </head>  0.031638
7   <metahttp-equiv="Content-Type"content="text/ht...  0.011219
0    

(<IPython.core.display.HTML object>,
 <IPython.core.display.HTML object>,
 ['</title>',
  '<bodybgcolor="#FFFFFF"background="foto/tile9.gif"link="#FFFFFF"vlink="#FFFFFF"alink="#FFFFFF"leftmargin="5"topmargin="5"onLoad="">',
  '</head>',
  '<metahttp-equiv="Content-Type"content="text/html;charset=utf-8">',
  '<HTML>',
  '<head>',
  '<divalign="center">',
  '<scripttype="text/javascript"src="stm31.js"></script>',
  '<divalign="center"><ahref="http://www.nedstat.nl/cgi-bin/viewstat?name=telkas"target="_new"><imgsrc="http://www.nedstat.nl/cgi-bin/nedstat.gif?name=telkas"border="0"width="16"height="16"nosave></a>',
  '<tdheight="179">'],
                                                 Token     Score  \
 0                                              <HTML>  0.007802   
 1                                              <head>  0.006601   
 2                     <title>HomepagevanKasperenGytha  0.002295   
 3                                            </title>  0.068092   
 4   <scripttype="t

#### 3.2.2. 1997–1999 $\rightarrow$ 2007–2010

In [11]:
# Load the sample subset (n=3)
conf_97_07 = missclassifications(model_v2_3, tokenizer_v2_3, df_13, 0, 3)

Calculating wrong predictions:   2%|▊                                                  | 7/455 [00:05<06:19,  1.18it/s]


In [12]:
visualization(model_v2_3, tokenizer_v2_3, conf_97_07, item_index=0)

Top 10 attributed tokens: ['<metaname="ProgId"content="FrontPage.Editor.Document">', '</title>">', '</head>', '</a', '<head>', '<framename="onder"src="start1.htm"target="_self">', '<ahref="mailto:Jeanmi@mobistar.be">Jean-MichelBuess', '<title>Reto2002', '<framesetrows="13%,*"framespacing="0"border="0"frameborder="0">', '<p>Opdezepaginawordenframesgebruikt,maaruwbrowserondersteuntgeenframes.']

URL: www.reto2002.home.xs4all.nl
True label: 19971999

=== ORIGINAL PAYLOAD ===
Top Positive Tokens:
                                                Token     Score
5   <metaname="ProgId"content="FrontPage.Editor.Do...  0.101491
4                                          </title>">  0.033572
7                                             </head>  0.025408
18                                                </a  0.012032
1                                              <head>  0.011267
10  <framename="onder"src="start1.htm"target="_self">  0.008992
17  <ahref="mailto:Jeanmi@mobistar.be">Jean-Michel... 

(<IPython.core.display.HTML object>,
 <IPython.core.display.HTML object>,
 ['<metaname="ProgId"content="FrontPage.Editor.Document">',
  '</title>">',
  '</head>',
  '</a',
  '<head>',
  '<framename="onder"src="start1.htm"target="_self">',
  '<ahref="mailto:Jeanmi@mobistar.be">Jean-MichelBuess',
  '<title>Reto2002',
  '<framesetrows="13%,*"framespacing="0"border="0"frameborder="0">',
  '<p>Opdezepaginawordenframesgebruikt,maaruwbrowserondersteuntgeenframes.'],
                                                 Token     Score  \
 0                                              <html> -0.006141   
 1                                              <head>  0.011267   
 2   <metahttp-equiv="Content-Type"content="text/ht... -0.012876   
 3                                     <title>Reto2002  0.003072   
 4                                          </title>">  0.033572   
 5   <metaname="ProgId"content="FrontPage.Editor.Do...  0.101491   
 6     <metaname="MicrosoftBorder"content="b,default"> -0.00

In [ ]:
visualization(model_v2_3, tokenizer_v2_3, conf_97_07, item_index=1)

Top 10 attributed tokens: ['<metaname="ProgId"content="FrontPage.Editor.Document">', '</h3>', '<li>', '</a><br>', '<title>pdvsfh', '</title>">', '<ul>', '<small>', '<head>', '</a><br>']

URL: www.pdvsfh.home.xs4all.nl
True label: 19971999

=== ORIGINAL PAYLOAD ===
Top Positive Tokens:
                                                Token     Score
4   <metaname="ProgId"content="FrontPage.Editor.Do...  0.119064
13                                              </h3>  0.039823
11                                               <li>  0.029702
26                                           </a><br>  0.017222
2                                       <title>pdvsfh  0.015722
3                                          </title>">  0.015691
10                                               <ul>  0.012022
7                                             <small>  0.011134
1                                              <head>  0.010151
22                                           </a><br>  0.007930

Top Negat

(<IPython.core.display.HTML object>,
 <IPython.core.display.HTML object>,
 ['<metaname="ProgId"content="FrontPage.Editor.Document">',
  '</h3>',
  '<li>',
  '</a><br>',
  '<title>pdvsfh',
  '</title>">',
  '<ul>',
  '<small>',
  '<head>',
  '</a><br>'],
                                                 Token     Score  \
 0                                              <html> -0.032031   
 1                                              <head>  0.010151   
 2                                       <title>pdvsfh  0.015722   
 3                                          </title>">  0.015691   
 4   <metaname="ProgId"content="FrontPage.Editor.Do...  0.119064   
 5                                             </head> -0.002349   
 6   <bodybgcolor="#000000"text="#ffffff"link="#dde...  0.004692   
 7                                             <small>  0.011134   
 8   <h2><fontcolor="red">PauldeVries&amp;FiaHuizin...  0.002481   
 9                                        </font></h2>  0.006684  

In [ ]:
visualization(model_v2_3, tokenizer_v2_3, conf_97_07, item_index=2)

Top 10 attributed tokens: ['</script>', '</script>', '<!--mstheme--><linkrel="stylesheet"type="text/css"href="_themes/bubbles/bubb1111.css"><metaname="MicrosoftTheme"content="bubbles1111,default">', '<metaname="ProgId"content="FrontPage.Editor.Document">', '<scriptlanguage="JavaScript"fptype="dynamicanimation">', '</title>">', '</head>', '<metahttp-equiv="Content-Language"content="nl">', '<!--functiondynAnimation(){}functionclickSwapImg(){}//-->', '<bodyonload="dynAnimation()"><!--msnavigation--><tableborder="0"cellpadding="0"cellspacing="0"width="100%"><tr><td>']

URL: www.nassaufa.home.xs4all.nl
True label: 19971999

=== ORIGINAL PAYLOAD ===
Top Positive Tokens:
                                                Token     Score
11                                          </script>  0.058618
9                                           </script>  0.041226
12  <!--mstheme--><linkrel="stylesheet"type="text/...  0.020695
6   <metaname="ProgId"content="FrontPage.Editor.Do...  0.017522
7   <sc

(<IPython.core.display.HTML object>,
 <IPython.core.display.HTML object>,
 ['</script>',
  '</script>',
  '<!--mstheme--><linkrel="stylesheet"type="text/css"href="_themes/bubbles/bubb1111.css"><metaname="MicrosoftTheme"content="bubbles1111,default">',
  '<metaname="ProgId"content="FrontPage.Editor.Document">',
  '<scriptlanguage="JavaScript"fptype="dynamicanimation">',
  '</title>">',
  '</head>',
  '<metahttp-equiv="Content-Language"content="nl">',
  '<!--functiondynAnimation(){}functionclickSwapImg(){}//-->',
  '<bodyonload="dynAnimation()"><!--msnavigation--><tableborder="0"cellpadding="0"cellspacing="0"width="100%"><tr><td>'],
                                                 Token     Score  \
 0                                              <html>  0.001610   
 1                                              <head>  0.001478   
 2   <metahttp-equiv="Content-Type"content="text/ht... -0.000607   
 3     <metahttp-equiv="Content-Language"content="nl">  0.002905   
 4                   

#### 3.2.3. 2003–2006 $\rightarrow$ 1997–1999 (41 instances)
Why are mid-2000s pages leaking into the 90s bucket?

In [27]:
# Load the sample subset (n=3)
conf_03_97 = missclassifications(model_v2_3, tokenizer_v2_3, df_12, 2, 0)

Calculating wrong predictions:  27%|██▋       | 25/93 [00:02<00:06, 10.75it/s]


In [23]:
visualization(model_v2_3, tokenizer_v2_3, conf_03_97, item_index=0)

Top 10 attributed tokens: ['</TITLE>', '<HEAD>', '<metaname="MicrosoftBorder"content="tl">', '<TITLE>TheFonzBierPage', '<html>', '<METANAME="Description"CONTENT="BierdatabasevanNederlandenBelgi,enderestgroeit.EHBOafdelingLexmond.GenealogieBrandenWillers">', '<metaname="MicrosoftTheme"content="neon0111">', '<METANAME="Keywords"CONTENT="Bier,Beer,EHBOLexmond,GenealogieBrandenWillers">', '</head>', '<bgsoundsrc="midi/cafe%20ad%20haven.mid"loop="-1">']

URL: www.ahcb.home.xs4all.nl
True label: 20032006

=== ORIGINAL PAYLOAD ===
Top Positive Tokens:
                                                Token     Score
3                                            </TITLE>  0.146924
1                                              <HEAD>  0.068493
10           <metaname="MicrosoftBorder"content="tl">  0.056396
2                              <TITLE>TheFonzBierPage  0.054740
0                                              <html>  0.052158
5   <METANAME="Description"CONTENT="Bierdatabaseva...  0.021676
9

(<IPython.core.display.HTML object>,
 <IPython.core.display.HTML object>,
 ['</TITLE>',
  '<HEAD>',
  '<metaname="MicrosoftBorder"content="tl">',
  '<TITLE>TheFonzBierPage',
  '<html>',
  '<METANAME="Description"CONTENT="BierdatabasevanNederlandenBelgi,enderestgroeit.EHBOafdelingLexmond.GenealogieBrandenWillers">',
  '<metaname="MicrosoftTheme"content="neon0111">',
  '<METANAME="Keywords"CONTENT="Bier,Beer,EHBOLexmond,GenealogieBrandenWillers">',
  '</head>',
  '<bgsoundsrc="midi/cafe%20ad%20haven.mid"loop="-1">'],
                                                 Token     Score  \
 0                                              <html>  0.052158   
 1                                              <HEAD>  0.068493   
 2                              <TITLE>TheFonzBierPage  0.054740   
 3                                            </TITLE>  0.146924   
 4   <METANAME="Keywords"CONTENT="Bier,Beer,EHBOLex...  0.011427   
 5   <METANAME="Description"CONTENT="Bierdatabaseva...  0.021676   
 6 

In [30]:
visualization(model_v2_3, tokenizer_v2_3, conf_03_97, item_index=3)

Top 10 attributed tokens: ['</TITLE>', '</HEAD>', '<TABLEHEIGHT=100%BORDER=0>', '<HEAD>', '<BODYSCROLLING=NO>', '<TITLE>DivX-files:YouronlinesourceforDivx-files', '<HTML>', '<!--if(window.Event)document.captureEvents(Event.MOUSEUP);functionnocontextmenu(){event.cancelBubble=trueevent.returnValue=false;returnfalse;}functionnorightclick(e){if(window.Event){if(e.which==2||e.which==3)returnfalse;}elseif(event.button==2||event.button==3){event.cancelBubble=trueevent.returnValue=false;returnfalse;}}document.oncontextmenu=nocontextmenu;document.onmousedown=norightclick;//-->', '<MAPNAME="enter">', '<AREAHREF="enterindex.html"ALT="EnterDutch"SHAPE="rect"COORDS="208']

URL: www.stuuropw.home.xs4all.nl
True label: 20032006

=== ORIGINAL PAYLOAD ===
Top Positive Tokens:
                                                Token     Score
3                                            </TITLE>  0.067425
7                                             </HEAD>  0.054846
11                         <TABLEHEIGH

(<IPython.core.display.HTML object>,
 <IPython.core.display.HTML object>,
 ['</TITLE>',
  '</HEAD>',
  '<TABLEHEIGHT=100%BORDER=0>',
  '<HEAD>',
  '<BODYSCROLLING=NO>',
  '<TITLE>DivX-files:YouronlinesourceforDivx-files',
  '<HTML>',
  '<!--if(window.Event)document.captureEvents(Event.MOUSEUP);functionnocontextmenu(){event.cancelBubble=trueevent.returnValue=false;returnfalse;}functionnorightclick(e){if(window.Event){if(e.which==2||e.which==3)returnfalse;}elseif(event.button==2||event.button==3){event.cancelBubble=trueevent.returnValue=false;returnfalse;}}document.oncontextmenu=nocontextmenu;document.onmousedown=norightclick;//-->',
  '<MAPNAME="enter">',
  '<AREAHREF="enterindex.html"ALT="EnterDutch"SHAPE="rect"COORDS="208'],
                                                 Token     Score  \
 0                                              <HTML>  0.005342   
 1                                              <HEAD>  0.022663   
 2     <TITLE>DivX-files:YouronlinesourceforDivx-files  0.00

In [24]:
visualization(model_v2_3, tokenizer_v2_3, conf_03_97, item_index=4)

Top 10 attributed tokens: ['</SCRIPT>', '</TITLE>', '<BODYBGCOLOR="#CCCCCC">', '<SCRIPTLANGUAGE="JavaScript">', '<HEAD>', '<MAPname="TOPCOORDS">', '</HEAD>', '<areaSHAPE=RECTCOORDS="129,0248,29"HREF="about.html">', '<areaSHAPE=RECTCOORDS="249,0415,29"HREF="products.html">', '<areaSHAPE=REC']

URL: www.bas0.home.xs4all.nl
True label: 20032006

=== ORIGINAL PAYLOAD ===
Top Positive Tokens:
                                                Token     Score
8                                           </SCRIPT>  0.186002
5                                            </TITLE>  0.088543
10                            <BODYBGCOLOR="#CCCCCC">  0.059250
6                       <SCRIPTLANGUAGE="JavaScript">  0.031493
1                                              <HEAD>  0.021693
11                              <MAPname="TOPCOORDS">  0.021404
9                                             </HEAD>  0.019240
13  <areaSHAPE=RECTCOORDS="129,0248,29"HREF="about...  0.014010
14  <areaSHAPE=RECTCOORDS="249,04

(<IPython.core.display.HTML object>,
 <IPython.core.display.HTML object>,
 ['</SCRIPT>',
  '</TITLE>',
  '<BODYBGCOLOR="#CCCCCC">',
  '<SCRIPTLANGUAGE="JavaScript">',
  '<HEAD>',
  '<MAPname="TOPCOORDS">',
  '</HEAD>',
  '<areaSHAPE=RECTCOORDS="129,0248,29"HREF="about.html">',
  '<areaSHAPE=RECTCOORDS="249,0415,29"HREF="products.html">',
  '<areaSHAPE=REC'],
                                                 Token     Score  \
 0                                              <HTML>  0.004539   
 1                                              <HEAD>  0.021693   
 2   <metaname=keywordscontent="DigitalFluff,Mobile...  0.007721   
 3   <metaname=descriptioncontent="DigitalFluffdeve... -0.001981   
 4   <TITLE>DigitalFluff:mobilephoneentertainmentan... -0.004555   
 5                                            </TITLE>  0.088543   
 6                       <SCRIPTLANGUAGE="JavaScript">  0.031493   
 7   <!--functionhandleform2(){vartheURL="play.html...  0.008581   
 8                         